# Task 3 perturbation-response playground

Task 3 is easy to misread because a mutant can look globally similar to WT while still having a very specific response that matters for scoring. This notebook starts from the public matched WT/Mab21l2 pair and deliberately changes only the mean perturbation response.

These are target-aware teaching controls, not held-out predictions. The transformations are defined once in [`experiments.py`](experiments.py) and reused by the committed result generator.

In [1]:
!pip -q install "git+https://github.com/aristoteleo/veckit.git@46d41e63f42a9aab815db20b742feeccd249cb17" pandas matplotlib

In [2]:
from pathlib import Path
import sys, urllib.request
import anndata as ad
import pandas as pd

LAB = Path.cwd() / 'intuition-lab'
if not (LAB / 'experiments.py').exists(): LAB = Path.cwd()
sys.path.insert(0, str(LAB))
from experiments import make_t3_failures, score_t3_failures

DATA = Path('/content/vec_t3_failures')
DATA.mkdir(exist_ok=True)
base = 'https://raw.githubusercontent.com/aristoteleo/veckit/46d41e63f42a9aab815db20b742feeccd249cb17/data/'
for name in ['sample_wt.h5ad', 'sample_mab21l2_ko.h5ad']:
    p = DATA / name
    if not p.exists(): urllib.request.urlretrieve(base + name, p)

wt_path = DATA / 'sample_wt.h5ad'
ko_path = DATA / 'sample_mab21l2_ko.h5ad'
wt, ko = ad.read_h5ad(wt_path), ad.read_h5ad(ko_path)
print('WT:', wt.shape, '| KO:', ko.shape)

WT: (150, 500) | KO: (150, 500)


## Change only the response

Let the known public mean response be `delta = mean(KO) - mean(WT)`. We create predictions with 0%, 25%, 50%, 100%, 150%, and 200% of that response, plus a reversed response and a gene-shuffled response.

The interesting comparison is between absolute similarity to the mutant and whether the **change from WT** points in the right direction with the right magnitude.

In [3]:
preds, meta = make_t3_failures(wt, ko, DATA / 'predictions', seed=0)
print('alphas:', meta['alphas'])
print('built:', list(preds))

alphas: [0.0, 0.25, 0.5, 1.0, 1.5, 2.0]
built: ['alpha_0', 'alpha_0.25', 'alpha_0.5', 'alpha_1', 'alpha_1.5', 'alpha_2', 'reversed', 'shuffled']


In [4]:
scores = score_t3_failures(preds, target=ko_path, wt=wt_path)
scores[['de_score', 'de_direction', 'severity_slope', 'mmd_u', 'variogram', 'absolute_pb_pearson']].round(4)

/opt/hostedtoolcache/Python/3.11.16/x64/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


/opt/hostedtoolcache/Python/3.11.16/x64/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


/opt/hostedtoolcache/Python/3.11.16/x64/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


/opt/hostedtoolcache/Python/3.11.16/x64/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


/opt/hostedtoolcache/Python/3.11.16/x64/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


/opt/hostedtoolcache/Python/3.11.16/x64/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


/opt/hostedtoolcache/Python/3.11.16/x64/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


/opt/hostedtoolcache/Python/3.11.16/x64/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,de_score,de_direction,severity_slope,mmd_u,variogram,absolute_pb_pearson
prediction,,,,,,
alpha_0,0.0000,0.0000,-6.9078,0.0462,0.0326,0.9033
alpha_0.25,0.6154,0.9984,-1.4921,0.0418,0.0158,0.9394
alpha_0.5,0.6154,0.9984,-0.7990,0.0390,0.0285,0.9654
alpha_1,0.6154,0.9984,-0.1059,0.0386,0.0668,0.9910
alpha_1.5,0.6154,0.9984,0.2996,0.0452,0.1128,0.9907
alpha_2,0.6154,0.9984,0.5873,0.0587,0.1629,0.9753
reversed,-0.1282,-0.9525,-6.9078,0.0658,0.1061,0.7987
shuffled,-0.0769,-0.0101,-6.9078,0.0445,0.1429,0.8386


In [5]:
sweep = scores.loc[[f'alpha_{a:g}' for a in meta['alphas']], ['de_score', 'de_direction', 'severity_slope', 'absolute_pb_pearson']].copy()
sweep.index = meta['alphas']
sweep.index.name = 'response scale alpha'
sweep.round(4)

,de_score,de_direction,severity_slope,absolute_pb_pearson
response scale alpha,,,,
0.00,0.0000,0.0000,-6.9078,0.9033
0.25,0.6154,0.9984,-1.4921,0.9394
0.50,0.6154,0.9984,-0.7990,0.9654
1.00,0.6154,0.9984,-0.1059,0.9910
1.50,0.6154,0.9984,0.2996,0.9907
2.00,0.6154,0.9984,0.5873,0.9753


## What I would remember

The zero-response control is the useful sanity check. It can retain high absolute pseudobulk correlation simply because WT and KO share a lot of baseline expression, while completely missing the knockout effect. Direction and magnitude are separate failure modes too: getting the right genes to move the right way does not guarantee that the response is strong enough.